# Gated-Fusion-guided GAN: Wasserstein distance versus forward KL

This notebook compares two adversarial objectives while holding the generator, conditional critic, pretrained Gated-Fusion dose classifier, data split, optimizer, random seed and training budget constant.

- **WGAN-GP** estimates Wasserstein-1 distance with a gradient penalty.
- **KL-fGAN** uses the variational forward-KL objective from the f-GAN framework: `E_real[T] - E_fake[exp(T - 1)]`.

Forward KL is not the loss of a conventional GAN. Calling the second run an f-GAN is important: its critic estimates a density-ratio-related variational bound rather than a probability of being real.

## Why this comparison is controlled

Both generators receive the same dose embedding and the same latent-noise stream. A separately loaded, frozen copy of the validation-winning `extended_cnn_gated_fusion` classifier supplies the same auxiliary dose loss in both runs. Freezing it prevents the reference decision boundary from drifting differently between objectives. Test data remains unopened until both models are trained.

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

from deeplearning_examples.io import load_data
from pytorch_cdgan.model import Generator, initialize_weights
from pytorch_timecourse_classification.artifacts import artifact_dir_for, load_model, load_selection_manifest

RANDOM_SEED = 37
NOISE_SIZE = 32
CONDITION_DIM = 8
EPOCHS = int(os.getenv("GAN_EPOCHS", "300"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, epochs per objective={EPOCHS}")

In [ ]:
selection = load_selection_manifest("extended_cnns_top3.json")
winner = min(selection["candidates"], key=lambda row: row["rank"])
checkpoint_path = artifact_dir_for(winner["experiment"]) / "model.pt"
reference_classifier, checkpoint = load_model(checkpoint_path, device=device)
assert checkpoint["model_config"]["architecture"] == "gated_fusion"
dose_names = np.asarray(checkpoint["class_names"], dtype=object)
dose_to_id = {dose: index for index, dose in enumerate(dose_names)}
signal_mean = float(checkpoint["signal_mean"])
signal_std = float(checkpoint["signal_std"])
reference_classifier.eval()
for parameter in reference_classifier.parameters():
    parameter.requires_grad_(False)
print(f"guide={winner['experiment']}, validation macro-F1={winner['macro_f1']:.4f}")

In [ ]:
train_trajectories, train_doses, _ = load_data("train")
train_labels = np.asarray([dose_to_id[str(dose)] for dose in train_doses], dtype=np.int64)
print(train_trajectories.shape, dict(zip(dose_names, np.bincount(train_labels))))

## 1. Shared conditional critic

Unlike the sigmoid discriminator in the introductory GAN notebook, this critic returns an unrestricted scalar. That is required by both Wasserstein and f-GAN variational objectives.

In [ ]:
class ConditionalCritic(nn.Module):
    def __init__(self, num_conditions: int, condition_dim: int = 8) -> None:
        super().__init__()
        self.condition_embedding = nn.Embedding(num_conditions, condition_dim)
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, 9, stride=2, padding=4), nn.LeakyReLU(0.2),
            nn.Conv1d(32, 64, 7, stride=2, padding=3), nn.LeakyReLU(0.2),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool1d(4), nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(96 * 4 + condition_dim, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, trajectories, labels):
        encoded = self.encoder(trajectories)
        conditioned = torch.cat([encoded, self.condition_embedding(labels.long())], dim=1)
        return self.head(conditioned)


def normalized_class_logits(classifier, raw_trajectories):
    return classifier((raw_trajectories - signal_mean) / signal_std)


def gradient_penalty(critic, real, fake, labels):
    alpha = torch.rand(len(real), 1, 1, device=real.device)
    interpolated = (alpha * real + (1.0 - alpha) * fake).requires_grad_(True)
    scores = critic(interpolated, labels)
    gradients = torch.autograd.grad(
        scores, interpolated, torch.ones_like(scores), create_graph=True, retain_graph=True
    )[0]
    return ((gradients.flatten(1).norm(2, dim=1) - 1.0) ** 2).mean()

## 2. Objective definitions

The KL conjugate contains an exponential and is numerically much less forgiving than Wasserstein loss. Its exponent is clipped to `[-10, 10]`; the reported result is therefore a stabilized approximation. This limitation is part of the comparison, not hidden implementation detail.

In [ ]:
def kl_conjugate(critic_scores):
    return torch.exp(torch.clamp(critic_scores - 1.0, min=-10.0, max=10.0))


def critic_objective(kind, critic, real, fake, labels, gradient_penalty_weight):
    real_scores = critic(real, labels)
    fake_scores = critic(fake, labels)
    if kind == "wasserstein":
        penalty = gradient_penalty(critic, real, fake, labels)
        loss = fake_scores.mean() - real_scores.mean() + gradient_penalty_weight * penalty
        estimate = real_scores.mean() - fake_scores.mean()
    elif kind == "forward_kl":
        penalty = torch.zeros((), device=real.device)
        # Negative of the variational lower bound: E_P[T] - E_Q[f*(T)].
        loss = -real_scores.mean() + kl_conjugate(fake_scores).mean()
        estimate = real_scores.mean() - kl_conjugate(fake_scores).mean()
    else:
        raise ValueError(f"unknown objective: {kind}")
    return loss, estimate.detach(), penalty.detach()


def generator_distance_loss(kind, fake_scores):
    if kind == "wasserstein":
        return -fake_scores.mean()
    if kind == "forward_kl":
        # Minimize the optimized f-GAN value; the real-data term is constant for G.
        return -kl_conjugate(fake_scores).mean()
    raise ValueError(f"unknown objective: {kind}")

In [ ]:
@dataclass(frozen=True)
class DistanceGanConfig:
    epochs: int = 300
    batch_size: int = 256
    critic_steps: int = 3
    learning_rate: float = 1e-4
    gradient_penalty_weight: float = 10.0
    classifier_weight: float = 0.25
    classifier_warmup_epochs: int = 50
    report_every: int = 25


def set_trainable(module, trainable):
    for parameter in module.parameters():
        parameter.requires_grad_(trainable)


def train_distance_gan(kind, generator, critic, trajectories, labels, *, config, device):
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    tensor = torch.as_tensor(trajectories[:, None, :], dtype=torch.float32)
    targets = torch.as_tensor(labels, dtype=torch.long)
    loader = DataLoader(
        TensorDataset(tensor, targets), batch_size=config.batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_SEED),
    )
    generator, critic = generator.to(device), critic.to(device)
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=config.learning_rate, betas=(0.0, 0.9))
    c_optimizer = torch.optim.Adam(critic.parameters(), lr=config.learning_rate, betas=(0.0, 0.9))
    history = {key: [] for key in ("critic_loss", "generator_loss", "distance_estimate", "gradient_penalty", "dose_accuracy")}

    for epoch in range(1, config.epochs + 1):
        totals = np.zeros(5)
        seen = 0
        for real, condition in loader:
            real, condition = real.to(device), condition.to(device)
            batch_size = len(real)
            for _ in range(config.critic_steps):
                set_trainable(critic, True)
                c_optimizer.zero_grad(set_to_none=True)
                with torch.no_grad():
                    fake = generator(torch.randn(batch_size, NOISE_SIZE, device=device), condition)
                c_loss, estimate, penalty = critic_objective(
                    kind, critic, real, fake, condition, config.gradient_penalty_weight
                )
                c_loss.backward()
                nn.utils.clip_grad_norm_(critic.parameters(), 10.0)
                c_optimizer.step()

            set_trainable(critic, False)
            g_optimizer.zero_grad(set_to_none=True)
            fake = generator(torch.randn(batch_size, NOISE_SIZE, device=device), condition)
            fake_scores = critic(fake, condition)
            dose_logits = normalized_class_logits(reference_classifier, fake)
            guide_weight = config.classifier_weight * min(1.0, epoch / max(config.classifier_warmup_epochs, 1))
            g_loss = generator_distance_loss(kind, fake_scores) + guide_weight * F.cross_entropy(dose_logits, condition)
            g_loss.backward()
            nn.utils.clip_grad_norm_(generator.parameters(), 10.0)
            g_optimizer.step()

            metrics = np.asarray([
                c_loss.item(), g_loss.item(), estimate.item(), penalty.item(),
                (dose_logits.argmax(1) == condition).float().mean().item(),
            ])
            totals += metrics * batch_size
            seen += batch_size
        set_trainable(critic, True)
        for key, value in zip(history, totals / seen):
            history[key].append(float(value))
        if epoch == 1 or epoch % config.report_every == 0 or epoch == config.epochs:
            print(
                f"{kind:11s} epoch {epoch:4d}: C={history['critic_loss'][-1]:.3f}, "
                f"G={history['generator_loss'][-1]:.3f}, estimate={history['distance_estimate'][-1]:.3f}, "
                f"dose acc={history['dose_accuracy'][-1]:.3f}"
            )
    generator.eval(); critic.eval()
    return generator, critic, history

## 3. Train both objectives

Set `GAN_EPOCHS=1` before starting the kernel for a quick end-to-end check. Full comparison uses 300 epochs and should be repeated with multiple seeds before drawing a final conclusion.

In [ ]:
config = DistanceGanConfig(epochs=EPOCHS)
trained = {}
for display_name, objective in (("WGAN-GP", "wasserstein"), ("KL-fGAN", "forward_kl")):
    print(f"\n--- {display_name} ---")
    torch.manual_seed(RANDOM_SEED)
    generator = Generator(
        noise_size=NOISE_SIZE, output_length=train_trajectories.shape[1],
        num_conditions=len(dose_names), condition_dim=CONDITION_DIM,
    )
    critic = ConditionalCritic(len(dose_names), CONDITION_DIM)
    generator.apply(initialize_weights)
    critic.apply(initialize_weights)
    trained[display_name] = train_distance_gan(
        objective, generator, critic, train_trajectories, train_labels, config=config, device=device
    )

## 4. Diagnostics

The Wasserstein estimate and KL variational bound are on different scales and must not be compared numerically against one another. Their trajectories are useful only within their respective runs.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 11), sharex=True)
for name, (_, _, history) in trained.items():
    epochs = np.arange(1, len(history["generator_loss"]) + 1)
    axes[0].plot(epochs, history["generator_loss"], label=name)
    axes[1].plot(epochs, history["distance_estimate"], label=name)
    axes[2].plot(epochs, history["dose_accuracy"], label=name)
axes[0].set_ylabel("generator loss")
axes[1].set_ylabel("within-objective estimate")
axes[2].set(ylabel="generated dose accuracy", xlabel="epoch", ylim=(0, 1.02))
for axis in axes:
    axis.grid(alpha=0.2); axis.legend()
fig.suptitle("Training diagnostics (objective scales differ)")
fig.tight_layout()

## 5. Held-out test comparison

In [ ]:
test_trajectories, test_doses, _ = load_data("test")
test_labels = np.asarray([dose_to_id[str(dose)] for dose in test_doses], dtype=np.int64)
test_counts = np.bincount(test_labels, minlength=len(dose_names))

def sample_by_dose(model, counts, random_seed=RANDOM_SEED + 1):
    random = torch.Generator(device=device).manual_seed(random_seed)
    samples, labels = [], []
    with torch.no_grad():
        for dose_id, count in enumerate(counts):
            condition = torch.full((int(count),), dose_id, dtype=torch.long, device=device)
            noise = torch.randn(int(count), NOISE_SIZE, generator=random, device=device)
            samples.append(model(noise, condition).squeeze(1).cpu().numpy())
            labels.append(np.full(int(count), dose_id))
    return np.concatenate(samples), np.concatenate(labels).astype(int)

generated = {name: sample_by_dose(model, test_counts) for name, (model, _, _) in trained.items()}

In [ ]:
FEATURES = ("mean signal", "temporal std", "maximum signal", "dynamic range", "endpoint change", "mean absolute step")

def trajectory_features(values, labels):
    frame = pd.DataFrame({"dose_id": labels})
    frame["mean signal"] = values.mean(1)
    frame["temporal std"] = values.std(1)
    frame["maximum signal"] = values.max(1)
    frame["dynamic range"] = np.ptp(values, axis=1)
    frame["endpoint change"] = values[:, -12:].mean(1) - values[:, :12].mean(1)
    frame["mean absolute step"] = np.abs(np.diff(values, axis=1)).mean(1)
    return frame

frames = {"test data": trajectory_features(test_trajectories, test_labels)}
frames.update({name: trajectory_features(values, labels) for name, (values, labels) in generated.items()})

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
positions = np.arange(len(dose_names))
styles = ((-0.18, "black"), (0.0, "tab:blue"), (0.18, "tab:orange"))
for feature, axis in zip(FEATURES, axes.flat):
    for ((name, frame), (offset, color)) in zip(frames.items(), styles):
        medians = frame.groupby("dose_id")[feature].median().reindex(range(len(dose_names)))
        axis.plot(positions + offset, medians, "o-", color=color, label=name)
    axis.set_title(feature); axis.set_xticks(positions, dose_names, rotation=35); axis.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle("Held-out median feature comparison: Wasserstein versus forward KL")
fig.tight_layout()

In [ ]:
def feature_nmae(real_frame, fake_frame, feature):
    real = real_frame.groupby("dose_id")[feature].median().reindex(range(len(dose_names))).to_numpy()
    fake = fake_frame.groupby("dose_id")[feature].median().reindex(range(len(dose_names))).to_numpy()
    return np.mean(np.abs(real - fake)) / max(float(np.ptp(real)), np.finfo(float).eps)

rows = []
for name, (samples, requested) in generated.items():
    normalized = torch.as_tensor(((samples - signal_mean) / signal_std)[:, None, :], dtype=torch.float32, device=device)
    with torch.no_grad():
        predicted = torch.cat([reference_classifier(batch).argmax(1).cpu() for batch in normalized.split(256)]).numpy()
    errors = {feature: feature_nmae(frames["test data"], frames[name], feature) for feature in FEATURES}
    rows.append({"variant": name, "dose fidelity": np.mean(predicted == requested), **errors, "mean feature NMAE": np.mean(list(errors.values()))})
comparison = pd.DataFrame(rows).set_index("variant").sort_values("mean feature NMAE")
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(len(dose_names), len(generated), figsize=(11, 18), sharex=True, sharey=True, squeeze=False)
rng = np.random.default_rng(RANDOM_SEED + 2)
for column, (name, (samples, labels)) in enumerate(generated.items()):
    for dose_id, dose in enumerate(dose_names):
        axis = axes[dose_id, column]
        real = test_trajectories[test_labels == dose_id]
        fake = samples[labels == dose_id]
        axis.plot(real[rng.choice(len(real), min(5, len(real)), replace=False)].T, color="black", alpha=0.14)
        axis.plot(fake[rng.choice(len(fake), min(5, len(fake)), replace=False)].T, alpha=0.65)
        if dose_id == 0: axis.set_title(name)
        if column == 0: axis.set_ylabel(str(dose))
        if dose_id == len(dose_names) - 1: axis.set_xlabel("time index")
fig.suptitle("Test trajectories (black) and generated trajectories")
fig.tight_layout()

## Result-specific interpretation

In [ ]:
best_feature_objective = comparison["mean feature NMAE"].idxmin()
best_dose_objective = comparison["dose fidelity"].idxmax()
print(
    f"On the held-out test fold, {best_feature_objective} has the lower mean feature NMAE "
    f"({comparison.loc[best_feature_objective, 'mean feature NMAE']:.3f}); "
    f"{best_dose_objective} has the higher frozen-classifier dose fidelity "
    f"({comparison.loc[best_dose_objective, 'dose fidelity']:.3f}).\n"
    f"Final generated-dose accuracies during training were "
    f"WGAN-GP={trained['WGAN-GP'][2]['dose_accuracy'][-1]:.3f} and "
    f"KL-fGAN={trained['KL-fGAN'][2]['dose_accuracy'][-1]:.3f}.\n"
    f"The distance estimates are intentionally not ranked across objectives because their scales differ. "
    f"The trajectory grid must still confirm that {best_feature_objective} retains within-dose diversity."
)

## Next experiments

- **projection discriminator + hinge loss:** strong conditional signal and usually simpler/stabler than an auxiliary classifier;
- **multi-scale or PatchGAN critics:** separately assess local bursts and global trajectory shape;
- **feature matching:** match Gated-Fusion embeddings or the six biological statistics to reduce classifier exploitation;
- **PacGAN/minibatch-standard-deviation layers:** expose low diversity to the critic and target mode collapse;
- **TCN or coarse spline generator:** encode smooth temporal structure directly;
- **WGAN-GP with several seeds and critic-step sweeps:** the most useful immediate follow-up because adversarial conclusions from one seed are fragile.